# Watershed delineation from a D8 flow grid

Choose a raster outlet and find all valid cells whose downstream D8 path reaches it. This is a contributing-area mask, not a flood extent.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/watershed-delineation'))
from watershed_delineation import delineate_watershed


## 1. Select an outlet cell

The synthetic direction grid contains two tributaries that meet at the selected outlet. Code 0 is a terminal cell; -1 is invalid.

In [ ]:
directions = np.array([
    [-1, 4, -1, -1, -1],
    [1, 1, 1, 4, -1],
    [-1, -1, 1, 1, 0],
] , dtype=int)
outlet = (2, 4)
basin = delineate_watershed(directions, outlet=outlet)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(np.ma.masked_where(directions == -1, directions), cmap='tab20')
axes[0].set_title('D8 direction codes')
axes[1].imshow(basin, cmap='Greens', vmin=0, vmax=1)
axes[1].scatter([outlet[1]], [outlet[0]], marker='*', s=180, label='Outlet')
axes[1].set_title('Contributing area'); axes[1].legend()
for ax in axes: ax.set_xticks(range(directions.shape[1])); ax.set_yticks(range(directions.shape[0]))
plt.tight_layout(); plt.show()
print('Cells in contributing area:', int(basin.sum()))


## 2. Move the outlet

The contributing area depends on the selected cell. Choose a different valid outlet and compare the mask.

In [ ]:
other_outlet = (1, 2)
other_basin = delineate_watershed(directions, outlet=other_outlet)
print('Original outlet cells:', int(basin.sum()))
print('Alternative outlet cells:', int(other_basin.sum()))
print(other_basin.astype(int))


## 3. Interpretation and checks

A watershed is a contributing-area result under a flow-routing model. It is not an inundation map. The direction grid must be hydrologically prepared, and the pour point must correspond to the intended outlet cell.

The implementation rejects cycles in the complete valid flow graph. It does not automatically snap an arbitrary point to a stream.

## Investigate

1. How does the mask change if you select an upstream cell?
2. What happens to cells draining to a separate sink?
3. Why should a pour-point snapping operation be explicit?
4. Why is the watershed boundary not a flood boundary?
5. How could D-infinity or multiple-flow-direction routing change the problem?